# LoomEar: 02. Pick-Cycle Synchronous Analysis & Residual Heatmaps
This notebook demonstrates the core innovation of LoomEar: folding acoustic waveforms into time-normalized cycles, constructing healthy templates, and extracting 2D residual error maps.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

from src.loomear.data.simulator import SyntheticLoomSimulator
from src.loomear.audio.preprocess import clean_audio
from src.loomear.audio.cycles import segment_cycles, build_healthy_template, compute_cycle_residual
from src.loomear.eval.explain import generate_residual_explanation

In [ ]:
sim = SyntheticLoomSimulator(sample_rate=16000, random_seed=42)
healthy_audio, _ = sim.generate_healthy_loom(duration_sec=4.0, rpm=212.0)
fault_audio = sim.inject_fault(healthy_audio, 'bearing_wear', severity=0.8, rpm=212.0)

res_healthy = segment_cycles(clean_audio(healthy_audio), sample_rate=16000)
res_fault = segment_cycles(clean_audio(fault_audio), sample_rate=16000)

print(f'Estimated Speed: {res_healthy.estimated_rpm:.1f} RPM, Confidence: {res_healthy.confidence*100:.1f}%')
print(f'Cycles Extracted: {len(res_healthy.cycles)}')

### Build Healthy Template & Compute Standardized Residual

In [ ]:
template = build_healthy_template(res_healthy.normalized_cycles, sample_rate=16000)
residual_map, score = compute_cycle_residual(res_fault.normalized_cycles[0], template, sample_rate=16000)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(residual_map, origin='lower', cmap='RdBu_r', aspect='auto')
ax.set_title(f'Standardized Residual Heatmap (Bearing Wear Anomaly Score: {score:.2f})')
ax.set_xlabel('Cycle Phase (%)')
ax.set_ylabel('Mel Frequency Bin')
plt.colorbar(im, ax=ax, label='Z-Score Residual')
plt.show()

exp = generate_residual_explanation(residual_map)
print('Mechanical Diagnosis:', exp['plain_english_summary'])